# Notebook 08 - Small VLMs and Multi-Image Workflows

This notebook shows when smaller vision-language models are enough and how to keep multi-image prompts organized.


## What you will learn

- how to compare large and small VLM footprints
- how to structure multi-image prompts with labels
- how to route simple image jobs away from heavier models


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate torch numpy pandas matplotlib Pillow
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "08_small_vlms_and_multi_image_workflows"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Compare small and medium VLM budgets

Routing is easier when the cost trade-off is visible.


In [ ]:
model_budgets = pd.DataFrame([
    {"model": "SmolVLM", "params_bucket": "small", "best_for": "single image or multi-image basics", "relative_cost": 1.0},
    {"model": "Qwen2.5-VL-3B", "params_bucket": "medium", "best_for": "stronger reasoning and video entry", "relative_cost": 2.8},
    {"model": "Qwen2.5-VL-7B", "params_bucket": "large", "best_for": "heavier reasoning", "relative_cost": 5.0},
])
model_budgets


## Step 2: Label every image in a multi-image prompt

Image labels reduce ambiguity once the conversation contains multiple visuals.


In [ ]:
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Compare image A and image B. Return the differences as bullet points."},
            {"type": "image", "path": "image_a.png", "label": "A"},
            {"type": "image", "path": "image_b.png", "label": "B"},
        ],
    }
]
print(json.dumps(conversation, indent=2))


## Step 3: Build a tiny router

Cheap tasks should stay cheap.


In [ ]:
def route_image_job(needs_video=False, image_count=1, requires_heavy_reasoning=False):
    if needs_video:
        return "Qwen2.5-VL-3B"
    if image_count <= 2 and not requires_heavy_reasoning:
        return "SmolVLM"
    return "Qwen2.5-VL-3B"

jobs = pd.DataFrame([
    {"job": "single screenshot classification", "route": route_image_job(image_count=1)},
    {"job": "two-image difference check", "route": route_image_job(image_count=2)},
    {"job": "dense report figure analysis", "route": route_image_job(image_count=1, requires_heavy_reasoning=True)},
])
jobs


## Exercises and extensions

1. Turn the router into a scoring function that also considers latency budgets.
1. Add a policy that rejects multi-image prompts once they exceed a maximum total patch budget.
